In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_R_K_Puram_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,380.0,182.0,214.0,118.0,207.0,227.0,NaN,44.0,95.0,171.0,381.0,315.0
1,2,373.0,262.0,110.0,139.0,229.0,159.0,112.0,65.0,79.0,198.0,360.0,311.0
2,3,368.0,238.0,165.0,139.0,302.0,167.0,109.0,61.0,81.0,152.0,393.0,284.0
3,4,395.0,311.0,177.0,164.0,297.0,215.0,72.0,52.0,62.0,178.0,401.0,208.0
4,5,369.0,227.0,130.0,171.0,279.0,243.0,78.0,46.0,68.0,163.0,390.0,192.0
5,6,350.0,178.0,162.0,183.0,257.0,165.0,60.0,49.0,80.0,133.0,365.0,212.0
6,7,354.0,216.0,201.0,164.0,294.0,260.0,49.0,45.0,60.0,137.0,393.0,256.0
7,8,376.0,188.0,163.0,169.0,243.0,238.0,52.0,46.0,101.0,144.0,397.0,322.0
8,9,391.0,120.0,154.0,231.0,191.0,187.0,90.0,54.0,142.0,182.0,374.0,215.0
9,10,310.0,316.0,201.0,202.0,213.0,177.0,149.0,60.0,105.0,131.0,363.0,314.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,380.000000,182.000000,214.0,118.000000,207.000000,227.00000,93.571429,44.0,95.000000,171.000000,381.000000,315.0
1,2,373.000000,262.000000,110.0,139.000000,229.000000,159.00000,112.000000,65.0,79.000000,198.000000,360.000000,311.0
2,3,368.000000,238.000000,165.0,139.000000,302.000000,167.00000,109.000000,61.0,81.000000,152.000000,393.000000,284.0
3,4,395.000000,311.000000,177.0,164.000000,297.000000,215.00000,72.000000,52.0,62.000000,178.000000,401.000000,208.0
4,5,369.000000,227.000000,130.0,171.000000,279.000000,243.00000,78.000000,46.0,68.000000,163.000000,390.000000,192.0
5,6,350.000000,178.000000,162.0,183.000000,257.000000,165.00000,60.000000,49.0,80.000000,133.000000,365.000000,212.0
6,7,354.000000,216.000000,201.0,164.000000,294.000000,160.46875,49.000000,45.0,60.000000,137.000000,393.000000,256.0
7,8,376.000000,188.000000,163.0,169.000000,243.000000,238.00000,52.000000,46.0,101.000000,144.000000,397.000000,322.0
8,9,391.000000,120.000000,154.0,231.000000,191.000000,187.00000,90.000000,54.0,142.000000,182.000000,374.000000,215.0
9,10,310.000000,316.000000,201.0,202.000000,213.000000,177.00000,149.000000,60.0,105.000000,131.000000,363.000000,314.0
